# Approach 1: Template-Based Modeling (TBM)\n# Stanford RNA 3D Folding Part 2\n\n## Pipeline\n1. MMseqs2 searches test sequences against the **entire PDB RNA** sequence database\n2. DasLab's `create_templates_csv.py` opens matching CIF files and copies C1' coordinates\n3. Output: `submission.csv` in competition format\n\n## Official Sources (no hallucination)\n- **DasLab create_templates**: https://github.com/DasLab/create_templates\n- **MMseqs2**: https://github.com/soedinglab/MMseqs2\n- **Official baseline notebook**: https://www.kaggle.com/code/rhijudas/mmseqs2-3d-rna-template-identification\n- **1st place TBM**: https://www.kaggle.com/code/jaejohn/rna-3d-folds-tbm-only-approach\n\n## IMPORTANT\nFor the **strongest** TBM result, fork the 1st-place notebook by jaejohn instead of running this.\nThis notebook uses the DasLab baseline pipeline for clarity and reproducibility.

In [ ]:
# ============================================================\n# CELL 1: Verify competition data\n# ============================================================\n# The competition provides the ENTIRE PDB RNA database at:\n#   /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA/\n#\n# This includes ~15,000+ CIF files, the sequence FASTA, and\n# release dates CSV. This is what your TRY0 was missing\n# (you only had 130 CIF files on Colab → 0 MMseqs2 hits).\n# ============================================================\nimport os, glob\n\nCOMP_DIR    = '/kaggle/input/stanford-rna-3d-folding-2'\nPDB_RNA_DIR = os.path.join(COMP_DIR, 'PDB_RNA')\nTEST_CSV    = os.path.join(COMP_DIR, 'test_sequences.csv')\nFASTA_DB    = os.path.join(PDB_RNA_DIR, 'pdb_seqres_NA.fasta')\nDATES_CSV   = os.path.join(PDB_RNA_DIR, 'pdb_release_dates_NA.csv')\n\nWORK_DIR = '/kaggle/working'\n\nprint('=== Data verification ===')\nfor label, path in [\n    ('Competition dir', COMP_DIR),\n    ('PDB_RNA dir',     PDB_RNA_DIR),\n    ('test_sequences',  TEST_CSV),\n    ('FASTA database',  FASTA_DB),\n    ('Release dates',   DATES_CSV),\n]:\n    exists = os.path.exists(path)\n    print(f'  {label}: {"OK" if exists else "MISSING"} — {path}')\n\ncif_files = glob.glob(os.path.join(PDB_RNA_DIR, '*.cif*'))\nprint(f'\\nCIF files: {len(cif_files):,}')\n\nif len(cif_files) == 0:\n    print('\\nERROR: No CIF files found.')\n    print('Make sure stanford-rna-3d-folding-2 competition data is attached.')\n    print('Click \"+ Add Input\" in the sidebar → search \"stanford-rna-3d-folding-2\"')

In [ ]:
# ============================================================\n# CELL 2: Load and inspect test sequences\n# ============================================================\nimport pandas as pd\n\ntest_df = pd.read_csv(TEST_CSV)\nprint(f'Test targets: {len(test_df)}')\nprint(f'Columns: {list(test_df.columns)}')\nprint(f'Sequence length range: {test_df["sequence"].str.len().min()} — {test_df["sequence"].str.len().max()}')\nprint()\nprint(test_df[['target_id', 'sequence']].head())

In [ ]:
# ============================================================\n# CELL 3: Install dependencies\n# ============================================================\n# MMseqs2: fast sequence search (like BLAST but much faster)\n#   Official: https://github.com/soedinglab/MMseqs2\n#\n# BioPython: for parsing CIF structure files\n#   Official: https://biopython.org/\n#\n# NOTE: For final submission with internet OFF, you need to\n# pre-install MMseqs2 as a Kaggle Dataset. See:\n#   https://www.kaggle.com/datasets/rhijudas/mmseqs2-binary\n# ============================================================\nimport subprocess, shutil\n\n# Check if mmseqs is already available\nif shutil.which('mmseqs') is None:\n    print('Installing MMseqs2...')\n    subprocess.run(['apt-get', 'update', '-qq'], check=False,\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    subprocess.run(['apt-get', 'install', '-y', '-qq', 'mmseqs2'],\n                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    print('MMseqs2 installed.')\nelse:\n    print('MMseqs2 already available.')\n\n!mmseqs version\n\n# BioPython (usually pre-installed on Kaggle, but install if missing)\ntry:\n    import Bio\n    print(f'BioPython {Bio.__version__} available.')\nexcept ImportError:\n    !pip install biopython --quiet\n    print('BioPython installed.')

In [ ]:
# ============================================================\n# CELL 4: Clone official DasLab create_templates repository\n# ============================================================\n# Source: https://github.com/DasLab/create_templates\n# License: Apache-2.0\n#\n# This repo contains create_templates_csv.py which does:\n#   - Parse MMseqs2 alignment results\n#   - Open matching CIF files from PDB_RNA/\n#   - Align query sequence to template sequence\n#   - Copy C1' atom coordinates from template to query\n#   - Handle insertions/deletions by interpolation\n#   - Output templates.csv in competition-compatible format\n# ============================================================\nTEMPLATES_REPO = os.path.join(WORK_DIR, 'create_templates')\n\nif not os.path.isdir(TEMPLATES_REPO):\n    !git clone https://github.com/DasLab/create_templates.git {TEMPLATES_REPO}\n    print(f'Cloned to {TEMPLATES_REPO}')\nelse:\n    print(f'Already cloned at {TEMPLATES_REPO}')\n\n# Verify the key script exists\nscript_path = os.path.join(TEMPLATES_REPO, 'create_templates_csv.py')\nassert os.path.isfile(script_path), f'Script not found: {script_path}'\nprint(f'create_templates_csv.py: OK')

In [ ]:
# ============================================================\n# CELL 5: Prepare query FASTA from test sequences\n# ============================================================\n# MMseqs2 requires input sequences in FASTA format.\n# We convert test_sequences.csv → query.fasta\n# ============================================================\nQUERY_FASTA = os.path.join(WORK_DIR, 'query.fasta')\n\nwith open(QUERY_FASTA, 'w') as f:\n    for _, row in test_df.iterrows():\n        f.write(f'>{row["target_id"]}\\n{row["sequence"]}\\n')\n\n# Verify\nwith open(QUERY_FASTA) as f:\n    lines = f.readlines()\nprint(f'Query FASTA: {len(lines)//2} sequences written to {QUERY_FASTA}')\nprint(f'First entry:')\nprint(f'  {lines[0].strip()}')\nprint(f'  {lines[1].strip()[:60]}...')

In [ ]:
# ============================================================\n# CELL 6: Run MMseqs2 search pipeline\n# ============================================================\n# MMseqs2 documentation: https://github.com/soedinglab/MMseqs2/wiki\n#\n# Step 1: createdb — convert FASTA files to MMseqs2 database format\n#   --dbtype 2 = nucleotide sequences\n#\n# Step 2: search — find similar sequences\n#   -s 7.5 = sensitivity (higher = finds more distant homologs)\n#   --search-type 3 = nucleotide search mode\n#   -e 10 = e-value threshold (larger = more permissive)\n#\n# Step 3: convertalis — convert binary results to readable TSV\n#   Output format: query,target,evalue,qstart,qend,tstart,tend,qaln,taln\n#   This is the exact format create_templates_csv.py expects\n#\n# These commands are documented in the DasLab create_templates README:\n#   https://github.com/DasLab/create_templates\n# ============================================================\nimport time\n\nQUERY_DB  = os.path.join(WORK_DIR, 'queryDB')\nTARGET_DB = os.path.join(WORK_DIR, 'targetDB')\nRESULT_DB = os.path.join(WORK_DIR, 'resultDB')\nTMP_DIR   = os.path.join(WORK_DIR, 'mmseqs_tmp')\nRESULT_TXT = os.path.join(WORK_DIR, 'Result.txt')\n\nos.makedirs(TMP_DIR, exist_ok=True)\n\n# Step 1: Build query database\nprint('Step 1/4: Building query database...')\nt0 = time.time()\n!mmseqs createdb {QUERY_FASTA} {QUERY_DB} --dbtype 2\nprint(f'  Done ({time.time()-t0:.1f}s)')\n\n# Step 2: Build target database from PDB RNA sequences\nprint('\\nStep 2/4: Building target database from PDB RNA...')\nt0 = time.time()\n!mmseqs createdb {FASTA_DB} {TARGET_DB} --dbtype 2\nprint(f'  Done ({time.time()-t0:.1f}s)')\n\n# Step 3: Search\nprint('\\nStep 3/4: Searching (this may take several minutes)...')\nt0 = time.time()\n!mmseqs search {QUERY_DB} {TARGET_DB} {RESULT_DB} {TMP_DIR} \\\n    -s 7.5 \\\n    --search-type 3 \\\n    -e 10\nprint(f'  Done ({time.time()-t0:.1f}s)')\n\n# Step 4: Convert results to TSV\nprint('\\nStep 4/4: Converting results...')\n# Format string matches what create_templates_csv.py expects\n# (documented in DasLab README)\n!mmseqs convertalis {QUERY_DB} {TARGET_DB} {RESULT_DB} {RESULT_TXT} \\\n    --format-output query,target,evalue,qstart,qend,tstart,tend,qaln,taln\n\n# Verify results\nif os.path.isfile(RESULT_TXT):\n    with open(RESULT_TXT) as f:\n        hits = f.readlines()\n    print(f'\\nTotal MMseqs2 hits: {len(hits)}')\n    \n    # Count unique queries with hits\n    queries_with_hits = set()\n    for line in hits:\n        queries_with_hits.add(line.split('\\t')[0])\n    print(f'Test targets with hits: {len(queries_with_hits)} / {len(test_df)}')\n    \n    if hits:\n        print(f'\\nFirst 3 hits:')\n        for line in hits[:3]:\n            parts = line.strip().split('\\t')\n            print(f'  Query={parts[0]:10s} Target={parts[1]:20s} E-value={parts[2]}')\nelse:\n    print('ERROR: No results file. Check MMseqs2 output above for errors.')

In [ ]:
# ============================================================\n# CELL 7: Run DasLab create_templates_csv.py\n# ============================================================\n# This is the official coordinate-transfer script from:\n#   https://github.com/DasLab/create_templates\n#\n# Arguments (from the repo's --help output):\n#   -s : CSV with target_id and sequence columns\n#   --mmseqs_results_file : MMseqs2 output (Result.txt from step 4)\n#   --cif_dir : directory containing CIF structure files\n#   --outfile : output CSV filename\n#   --max_templates : maximum templates per target (5 for competition)\n#   --skip_temporal_cutoff : skip date filtering (use for Part 2\n#     where temporal_cutoff column may differ)\n# ============================================================\nTEMPLATES_CSV = os.path.join(WORK_DIR, 'templates.csv')\n\nprint('Running create_templates_csv.py...')\nprint(f'  Sequences:  {TEST_CSV}')\nprint(f'  MMseqs2:    {RESULT_TXT}')\nprint(f'  CIF dir:    {PDB_RNA_DIR}')\nprint(f'  Output:     {TEMPLATES_CSV}')\nprint(f'  Max templates: 5')\nprint()\n\nt0 = time.time()\n!cd {TEMPLATES_REPO} && python3 create_templates_csv.py \\\n    -s {TEST_CSV} \\\n    --mmseqs_results_file {RESULT_TXT} \\\n    --cif_dir {PDB_RNA_DIR} \\\n    --outfile {TEMPLATES_CSV} \\\n    --max_templates 5 \\\n    --skip_temporal_cutoff\n\nelapsed = time.time() - t0\nprint(f'\\nCompleted in {elapsed:.1f}s')\n\nif os.path.isfile(TEMPLATES_CSV):\n    tmpl_df = pd.read_csv(TEMPLATES_CSV)\n    print(f'Templates output: {tmpl_df.shape}')\n    print(f'Columns: {list(tmpl_df.columns)[:10]}...')\nelse:\n    print('ERROR: templates.csv not generated. Check output above.')

In [ ]:
# ============================================================\n# CELL 8: Convert templates.csv to submission.csv\n# ============================================================\n# The DasLab create_templates_csv.py output format is similar\n# to labels.csv (the training data format). The competition\n# submission format requires specific columns:\n#\n#   ID, resname, resid, x_1,y_1,z_1, ..., x_5,y_5,z_5\n#\n# If the output already matches submission format, this cell\n# just copies it. If not, it reformats.\n#\n# Note: The official Kaggle notebooks handle this formatting\n# internally. If you forked one of those, skip this cell.\n# ============================================================\nimport numpy as np\n\nSUBMISSION_CSV = os.path.join(WORK_DIR, 'submission.csv')\n\nif os.path.isfile(TEMPLATES_CSV):\n    tmpl_df = pd.read_csv(TEMPLATES_CSV)\n    \n    # Check if already in submission format\n    expected_cols = ['ID', 'resname', 'resid', 'x_1', 'y_1', 'z_1']\n    if all(c in tmpl_df.columns for c in expected_cols):\n        # Already in submission format\n        tmpl_df.to_csv(SUBMISSION_CSV, index=False)\n        print(f'Output already in submission format.')\n    else:\n        # Need to reformat — inspect columns to determine mapping\n        print(f'Output columns: {list(tmpl_df.columns)}')\n        print(f'This output needs reformatting to submission format.')\n        print(f'Inspect the columns above and adapt the reformatting.')\n        print(f'\\nFor reference, submission format needs:')\n        print(f'  ID (target_resid), resname, resid, x_1..z_5')\n        # Copy as-is for now — manual inspection needed\n        tmpl_df.to_csv(SUBMISSION_CSV, index=False)\n    \n    print(f'\\nSubmission: {SUBMISSION_CSV}')\n    print(f'Shape: {tmpl_df.shape}')\nelse:\n    print('No templates.csv found. Cannot generate submission.')

In [ ]:
# ============================================================\n# CELL 9: Validate submission\n# ============================================================\nif os.path.isfile(SUBMISSION_CSV):\n    sub_df = pd.read_csv(SUBMISSION_CSV)\n    print(f'Submission shape: {sub_df.shape}')\n    print(f'Columns: {list(sub_df.columns)}')\n    print(f'\\nFirst 5 rows:')\n    print(sub_df.head())\n    \n    # Check coordinate ranges\n    coord_cols = [c for c in sub_df.columns if c.startswith(('x_', 'y_', 'z_'))]\n    if coord_cols:\n        all_coords = sub_df[coord_cols].values.flatten()\n        all_coords = all_coords[~np.isnan(all_coords)]\n        print(f'\\nCoordinate stats:')\n        print(f'  Range: [{all_coords.min():.1f}, {all_coords.max():.1f}]')\n        print(f'  Mean: {all_coords.mean():.1f}')\n        print(f'  Non-zero: {(all_coords != 0).sum()} / {len(all_coords)}')\n        \n        pct_nonzero = (all_coords != 0).sum() / len(all_coords) * 100\n        if pct_nonzero < 10:\n            print(f'\\nWARNING: Only {pct_nonzero:.1f}% non-zero coordinates.')\n            print(f'This suggests most targets had no template hits.')\n        else:\n            print(f'\\n{pct_nonzero:.1f}% non-zero coordinates — looks good!')\n    \n    print(f'\\nSubmission ready at: {SUBMISSION_CSV}')\nelse:\n    print('No submission file found.')

## Next steps\n\n1. **Download** `submission.csv` from the Output tab\n2. **Submit** to the competition\n3. **For stronger results**: Fork `jaejohn/rna-3d-folds-tbm-only-approach`\n4. **For best results**: Combine with Approach 2 (RibonanzaNet distance prediction)\n\n## For offline submission (internet OFF)\n\nThe competition requires notebooks to run with internet disabled.\nTo handle this:\n1. Pre-install MMseqs2 by uploading the binary as a Kaggle Dataset\n   (see https://www.kaggle.com/datasets/rhijudas/mmseqs2-binary)\n2. Clone DasLab/create_templates and upload as a Kaggle Dataset\n3. Reference both datasets in your notebook's Input section\n4. Update paths to use `/kaggle/input/your-dataset-name/...`